# Picking Equation Process - Simplified

This notebook consolidates equation coefficients and creates the final Excel file.

## Process
1. Copy coefficient files from MATLAB folder (already fitted)
2. Create consolidated Excel file with one sheet per equation

## Source
- MATLAB coefficient files: `C:\Users\NoahB\Documents\MATLAB\equation_coefficients\coefficient_files\`
- Example format: `Picking Equations\all_equations_coefficients_example.xlsx`

In [ ]:
# Cell 0: Setup and Configuration
from pathlib import Path
import pandas as pd
import shutil
from openpyxl import Workbook

# Paths
MATLAB_COEFF_DIR = Path(r'C:\Users\NoahB\Documents\MATLAB\equation_coefficients\coefficient_files')
PROJECT_ROOT = Path(r'C:\Users\NoahB\Documents\HebrewU Bioengineering\Cardiac_RODEO')
OUTPUT_DIR = PROJECT_ROOT / 'Picking Equations' / 'outputs'
FINAL_EXCEL = PROJECT_ROOT / 'EQN_Coefficients' / 'all_equations_coefficients.xlsx'

# All 11 equation names
EQUATIONS = [
    'polynomial',
    'modified_hill',
    'dual_exponential',
    'bivariate_gaussian',
    'gaussian_hill_hybrid',
    'gaussian_ridge',
    'pkpd_elimination',
    'adaptive_response',
    'recovery_model',
    'cumulative_exposure',
    'biphasic_response'
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'MATLAB coefficients: {MATLAB_COEFF_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Final Excel: {FINAL_EXCEL}')

In [ ]:
# Cell 1: Check available coefficient files in MATLAB folder
print('Available coefficient files in MATLAB folder:\n')

found_equations = []
missing_equations = []

for eq in EQUATIONS:
    csv_file = MATLAB_COEFF_DIR / f'{eq}_coefficients.csv'
    if csv_file.exists():
        found_equations.append(eq)
        print(f'  ✓ {eq}')
    else:
        missing_equations.append(eq)
        print(f'  ✗ {eq} (MISSING)')

print(f'\nFound: {len(found_equations)}/{len(EQUATIONS)} equations')

In [ ]:
# Cell 2: Copy coefficient files from MATLAB to local outputs folder
COPY_FILES = True  # Set to True to copy files

if COPY_FILES:
    print('Copying coefficient files from MATLAB...\n')
    
    for eq in found_equations:
        src = MATLAB_COEFF_DIR / f'{eq}_coefficients.csv'
        dst = OUTPUT_DIR / f'{eq}_coefficients.csv'
        shutil.copy2(src, dst)
        print(f'  Copied: {eq}_coefficients.csv')
    
    print(f'\nCopied {len(found_equations)} files to {OUTPUT_DIR}')
else:
    print('Skipping copy. Set COPY_FILES=True to copy.')

In [ ]:
# Cell 3: Create Excel file from coefficient CSVs
# This follows the exact same process as create_excel_from_coefficients.m

def create_excel_from_coefficients(coeff_dir, output_excel, equations):
    """
    Create Excel file with all equation coefficients.
    
    CSV format:
    # Equation: [Name]
    # [Formula]
    # Parameters: [list]
    #
    Drug,Contractility,,,,,,,,,,,,,,O2,,,,,,,,,
    Drug,Arrhythmia,Cardiotoxicity,Concern,R0,Emax,...,,R0,Emax,...
    [data rows]
    """
    print(f"{'='*80}")
    print('CREATING EXCEL FILE FROM COEFFICIENT CSVs')
    print(f"{'='*80}\n")
    
    coeff_dir = Path(coeff_dir)
    output_excel = Path(output_excel)
    
    # Delete existing file
    if output_excel.exists():
        output_excel.unlink()
        print(f'Deleted existing: {output_excel.name}')
    
    workbook = Workbook()
    if 'Sheet' in workbook.sheetnames:
        workbook.remove(workbook['Sheet'])
    
    success_count = 0
    
    for eq_name in equations:
        csv_file = coeff_dir / f'{eq_name}_coefficients.csv'
        
        if not csv_file.exists():
            print(f'  ✗ {eq_name}: file not found')
            continue
        
        try:
            # Read CSV as text
            with open(csv_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            
            # Find header row (contains Contractility or O2 or Arrhythmia)
            header_idx = None
            col_names_idx = None
            
            for i, line in enumerate(lines):
                stripped = line.strip()
                if stripped.startswith('#'):
                    continue
                if 'Contractility' in stripped or 'O2' in stripped or 'Arrhythmia' in stripped:
                    header_idx = i
                    col_names_idx = i + 1
                    break
            
            if col_names_idx is None:
                print(f'  ✗ {eq_name}: could not find header row')
                continue
            
            # Parse headers
            header_labels = [h.strip() for h in lines[header_idx].split(',')]
            col_names = [c.strip() for c in lines[col_names_idx].split(',')]
            
            # Parse data rows
            data_rows = []
            for line in lines[col_names_idx + 1:]:
                stripped = line.strip()
                if stripped:
                    data_rows.append([v.strip() for v in stripped.split(',')])
            
            if not data_rows:
                print(f'  ✗ {eq_name}: no data rows')
                continue
            
            # Create sheet
            sheet = workbook.create_sheet(title=eq_name)
            num_cols = len(col_names)
            
            # Row 1: Group headers (Contractility/O2)
            for c, label in enumerate(header_labels[:num_cols], 1):
                if label:
                    sheet.cell(1, c, label)
            
            # Row 2: Column names
            for c, name in enumerate(col_names, 1):
                sheet.cell(2, c, name)
            
            # Row 3+: Data
            for r, row_data in enumerate(data_rows, 3):
                for c, val in enumerate(row_data[:num_cols], 1):
                    if val:
                        try:
                            cell_val = float(val)
                        except ValueError:
                            cell_val = val
                        sheet.cell(r, c, cell_val)
            
            print(f'  ✓ {eq_name}: {len(data_rows)} drugs, {num_cols} columns')
            success_count += 1
            
        except Exception as e:
            print(f'  ✗ {eq_name}: {e}')
    
    # Save
    if workbook.sheetnames:
        workbook.active = workbook[workbook.sheetnames[0]]
        output_excel.parent.mkdir(parents=True, exist_ok=True)
        workbook.save(output_excel)
        
        print(f"\n{'='*80}")
        print(f'Excel file created: {output_excel}')
        print(f'Sheets: {success_count}/{len(equations)}')
        print(f"{'='*80}")
        return output_excel
    else:
        print('No sheets created!')
        return None

In [ ]:
# Cell 4: Run Excel creation
CREATE_EXCEL = True  # Set to True to create Excel

if CREATE_EXCEL:
    # Use OUTPUT_DIR (local copy) or MATLAB_COEFF_DIR (original)
    source_dir = OUTPUT_DIR if (OUTPUT_DIR / f'{EQUATIONS[0]}_coefficients.csv').exists() else MATLAB_COEFF_DIR
    
    result = create_excel_from_coefficients(source_dir, FINAL_EXCEL, EQUATIONS)
    
    if result:
        print(f'\nSuccess! Open: {result}')
else:
    print('Skipping Excel creation. Set CREATE_EXCEL=True to run.')

In [ ]:
# Cell 5: Verify Excel file matches expected format
if FINAL_EXCEL.exists():
    print('Verifying Excel file...\n')
    
    xl = pd.ExcelFile(FINAL_EXCEL)
    print(f'Sheets: {xl.sheet_names}\n')
    
    # Check first sheet
    first_sheet = xl.sheet_names[0]
    df = pd.read_excel(FINAL_EXCEL, sheet_name=first_sheet, header=None)
    
    print(f'Sheet: {first_sheet}')
    print(f'Row 0 (group headers): {df.iloc[0].tolist()[:10]}...')
    print(f'Row 1 (column names): {df.iloc[1].tolist()[:10]}...')
    print(f'Row 2 (first drug): {df.iloc[2].tolist()[:5]}...')
    print(f'\nTotal rows: {len(df)}')
else:
    print('Excel file not found. Run Cell 4 first.')

In [ ]:
# Cell 6: R² Summary across all equations
def summarize_r2(coeff_dir, equations):
    """Summarize R² values for all equations."""
    print(f"{'='*80}")
    print('R² SUMMARY ACROSS ALL EQUATIONS')
    print(f"{'='*80}\n")
    
    results = []
    
    for eq_name in equations:
        csv_file = Path(coeff_dir) / f'{eq_name}_coefficients.csv'
        if not csv_file.exists():
            continue
        
        try:
            # Read with header=1 to skip group header row
            # Find the actual header row first
            with open(csv_file, 'r') as f:
                lines = f.readlines()
            
            # Skip comments, find header
            skip_rows = 0
            for i, line in enumerate(lines):
                if line.startswith('#'):
                    skip_rows = i + 1
                elif 'Contractility' in line or 'Drug' in line:
                    skip_rows = i + 1  # Skip group header, use column names
                    break
            
            df = pd.read_csv(csv_file, skiprows=skip_rows)
            df.columns = df.columns.str.strip()
            
            # Find R2 columns
            r2_cols = [c for c in df.columns if 'R2' in c]
            if len(r2_cols) >= 1:
                r2_contract = pd.to_numeric(df[r2_cols[0]], errors='coerce').mean()
                r2_o2 = pd.to_numeric(df[r2_cols[1]], errors='coerce').mean() if len(r2_cols) > 1 else None
                
                results.append({
                    'equation': eq_name,
                    'n_drugs': len(df),
                    'contractility_r2_mean': r2_contract,
                    'o2_r2_mean': r2_o2
                })
        except Exception as e:
            print(f'  Error reading {eq_name}: {e}')
    
    if results:
        summary_df = pd.DataFrame(results)
        summary_df = summary_df.sort_values('contractility_r2_mean', ascending=False)
        display(summary_df)
        
        # Save summary
        summary_file = OUTPUT_DIR / 'all_equations_r2_summary.csv'
        summary_df.to_csv(summary_file, index=False)
        print(f'\nSaved: {summary_file}')
        return summary_df
    return None

# Run summary
source_dir = OUTPUT_DIR if (OUTPUT_DIR / f'{EQUATIONS[0]}_coefficients.csv').exists() else MATLAB_COEFF_DIR
r2_summary = summarize_r2(source_dir, EQUATIONS)